In [ ]:
# Buy signal:
# renko >= 2, 5-day OBV > 30 degrees
# exit if renko < 2

# Sell signal:
# renko <= -2, 5-day OBV < -30 degrees
# exit if renko > -2

In [33]:
import yfinance as yf
import pandas as pd
import numpy as np
import copy
import matplotlib.pyplot as plt
import statsmodels.api as sm
from stocktrends import Renko
import warnings

warnings.filterwarnings("ignore")

In [34]:
# average true range
def atr(DF, n=14):
    df = DF.copy()

    df["H-L"] = df["High"] - df["Low"]
    df["H-PC"] = df["High"] - df["Close"].shift(1)
    df["L-PC"] = df["Low"] - df["Close"].shift(1)

    # max along the rows between all three gives true range
    df["TR"] = df[["H-L", "H-PC", "L-PC"]].max(axis=1, skipna=False)

    df["ATR"] = df["TR"].ewm(com=n, min_periods=n).mean()
    return df["ATR"]

# slope of n consecutive points on a plot
def slope(ser, n): # ser is time series, n is window size
    slopes = [i * 0 for i in range(n-1)] # empty list of 0s, size n-1
    for i in range(n, len(ser)+1): # loop i over window endpoints
        y = ser[i-n:i] # last n points, dependent variable
        x = np.array(range(n)) # independent variable

        # scaling both to 0-1
        y_scaled = (y - y.min())/(y.max() - y.min())
        x_scaled = (x - x.min())/(x.max() - x.min())

        x_scaled = sm.add_constant(x_scaled)

        # fit linear regression
        model = sm.OLS(y_scaled, x_scaled) # OLS finds best-fitting straight line through set of data points
        results = model.fit()
        slopes.append(results.params[-1]) # result params contains [intercept, slope], so the [-1] grabs slope coefficient and appends to slopes
    slope_angle = np.rad2deg(np.arctan(np.array(slopes))) # convert to angle in degrees
    return np.array(slope_angle) # return 

# convert ohlc data to renko bricks
def renko_DF(DF):
    df = DF.copy()
    df.reset_index(inplace=True)
    df = df.iloc[:, [0, 1, 2, 3, 4, 5]]
    
    # renaming columns for Renko to work
    df.columns = ["date", "open", "high", "low", "close", "volume"]
    
    df2 = Renko(df)

    # compute atr, take most recent [-1], round to nearest integer and max(0.5, ...) enforces a minimum brick size
    df2.brick_size = max(0.5, round(atr(DF, 20).iloc[-1], 0))

    # generate bricks
    renko_df = df2.get_ohlc_data()

    # convert trend to numerical signal, +1 if uptrend, -1 it downtrend
    renko_df["bar_num"] = np.where(renko_df["uptrend"] == True, 1, np.where(renko_df["uptrend"] == False, -1, 0))

    # loop through "bar_num" numerical values
    for i in range(1, len(renko_df["bar_num"])):
        #if consecutive bricks in same direction, accumalate them
        if renko_df["bar_num"].loc[i] > 0 and renko_df["bar_num"].loc[i-1] > 0:
            renko_df["bar_num"][i] = renko_df["bar_num"].loc[i] + renko_df["bar_num"].loc[i-1]
        elif renko_df["bar_num"].loc[i] < 0 and renko_df["bar_num"].loc[i-1] < 0:
            renko_df["bar_num"][i] = renko_df["bar_num"].loc[i] + renko_df["bar_num"].loc[i-1]
    
    # remove duplicates and only keep last value
    renko_df.drop_duplicates(subset="date", keep="last", inplace=True)

    return renko_df # return

# obv measures buy/sell pressure 
# On Balance Volume
def obv(DF):
    df = DF.copy()
    df["daily_ret"] = df["Close"].pct_change()
    df["direction"] = np.where(df["daily_ret"]>=0, 1, -1)
    df["direction"][0] = 0
    df["vol_adj"] = df["Volume"] * df["direction"]
    df["obv"] = df["vol_adj"].cumsum()
    return df["obv"]

# Standard KPIs
def cagr(DF):
    df = DF.copy()
    df = df.copy()

    df["strat_return"] = df["Close"].pct_change()
    df["cum_return"] = (1 + df["strat_return"]).cumprod()

    # cum return ^ 1/n
    n = len(df)/(252*6.5) # 252 is trading days in a year
    # Using 1h candles. in one trading day theres 6.5 hours
    # if data in hours, divide n further by 8 (8 hours in trading day)

    CAGR = (df["cum_return"].tolist()[-1])**(1/n) - 1

    return CAGR


def volatility(DF):
    df = DF.copy()

    df["return"] = df["Close"].pct_change()

    # volatility is just st dev of returns
    # annualised volatility is volatility multiplied by sqrt(252)
    vol = df["return"].std() * np.sqrt(252*6.5)

    return vol


def sharpe(DF, rf):
    df = DF.copy()

    return (cagr(df) - rf)/volatility(df)


def max_dd(DF):
    df = DF.copy()

    df["return"] = df["Close"].pct_change()

    df["cum_return"] = 1 + df["return"].cumprod()

    df["cum_roll_max"] = df["cum_return"].cummax()

    df["drawdown"] = df["cum_roll_max"] - df["cum_return"]

    return (df["drawdown"]/df["cum_roll_max"]).max()



In [35]:
# Downloading data

tickers = ["MSFT", "AAPL", "FB", "AMZN", "INTC", "CSCO", "IBM", "TSLA", "AMD"]

ohlc_intraday = {}

for ticker in tickers:
    #ohlc_intraday[ticker] = yf.download(ticker, interval='1h', period='1y')
    df = yf.download(
        ticker,
        interval="1h",
        period="1y",
    )

    # 1) If yfinance gives MultiIndex columns, flatten them (keep OHLCV names)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # 2) If any duplicate columns exist (e.g., two "High"), remove duplicates
    df = df.loc[:, ~df.columns.duplicated()]

    #ohlc_intraday[ticker].dropna(inplace=True, how='all')
    # 3) Drop rows where everything is NaN
    df = df.dropna(how="all")

    #ohlc_intraday[ticker] = ohlc_intraday[ticker].between_time('09:30', '16:00')
    # 4) Keep regular session hours (works with DatetimeIndex)
    df = df.between_time("09:30", "16:00")

    ohlc_intraday[ticker] = df

# safety check
tickers = list(ohlc_intraday.keys())



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [ ]:
# merging the renko_df with original ohlc data
ohlc_renko = {}
df = copy.deepcopy(ohlc_intraday)

tickers_signal = {}
tickers_return = {}

for ticker in tickers:
    #print("merging for ", ticker)
    renko = renko_DF(df[ticker])
    df[ticker]["date"] = df[ticker].index

    # merging data sets, OUTER merge on DATE column
    df[ticker]["date"] = pd.to_datetime(df[ticker]["date"], utc=True)
    renko["date"] = pd.to_datetime(renko["date"], utc=True)
    ohlc_renko[ticker] = df[ticker].merge(renko.loc[:, ["date", "bar_num"]], how='outer', on='date')

    # forward filling NaN values
    ohlc_renko[ticker]["bar_num"].fillna(method='ffill', inplace=True)

    ohlc_renko[ticker]["obv"] = obv(ohlc_renko[ticker])
    ohlc_renko[ticker]["obv_slope"] = slope(ohlc_renko[ticker]["obv"], 5)
    tickers_signal[ticker] = ""
    tickers_return[ticker] = []

print(ohlc_renko)


merging for  MSFT
merging for  AAPL
merging for  FB
merging for  AMZN
merging for  INTC
merging for  CSCO
merging for  IBM
merging for  TSLA
merging for  AMD
{'MSFT':           Close        High         Low  ...  bar_num       obv  obv_slope
0    441.989990  446.880005  440.399994  ...      1.0         0   0.000000
1    442.418610  443.489990  441.457611  ...      1.0   1437427   0.000000
2    415.229889  422.859985  414.170013  ...     -4.0 -18408505   0.000000
3    415.250000  416.929993  413.160004  ...     -4.0 -11934512   0.000000
4    417.387390  420.679993  416.549988  ...     -4.0  -3818536 -22.949848
..          ...         ...         ...  ...      ...       ...        ...
662  472.230011  472.700012  469.480011  ...      3.0    707024  45.471488
663  476.109985  478.720001  473.160004  ...      3.0   5763146  43.987102
664  479.190002  479.799988  475.850006  ...      4.0   8102553  45.142116
665  481.019989  483.739990  479.290009  ...      5.0  11666478  45.618216
666  479

In [38]:
# identifying signals and calculating return

# recall
# Buy signal:
# renko >= 2, 5-day OBV > 30 degrees
# exit if renko < 2

# Sell signal:
# renko <= -2, 5-day OBV < -30 degrees
# exit if renko > -2

for ticker in tickers:
    for i in range(len(ohlc_intraday[ticker])):
        # no signal
        if tickers_signal[ticker] == "":
            tickers_return[ticker].append(0)
            if ohlc_renko[ticker]["bar_num"].iloc[i] >= 2 and ohlc_renko[ticker]["obv_slope"].iloc[i] > 30:
                tickers_signal[ticker] = "Buy"
            elif ohlc_renko[ticker]["bar_num"].iloc[i] <= -2 and ohlc_renko[ticker]["obv_slope"].iloc[i] < -30:
                tickers_signal[ticker] = "Sell"
        
        # buy signal
        elif tickers_signal[ticker] == "Buy":
            tickers_return[ticker].append((ohlc_renko[ticker]["Close"].iloc[i]/ohlc_renko[ticker]["Close"].iloc[i-1])-1)       
            if ohlc_renko[ticker]["bar_num"].iloc[i] <= 2 and ohlc_renko[ticker]["obv_slope"].iloc[i] < -30:
                tickers_signal[ticker] = "Sell"
            elif ohlc_renko[ticker]["bar_num"].iloc[i] < 2:
                tickers_signal[ticker] = ""

        # sell signal
        elif tickers_signal[ticker] == "Sell":
            tickers_return[ticker].append((ohlc_renko[ticker]["Close"].iloc[i-1]/ohlc_renko[ticker]["Close"].iloc[i])-1)       
            if ohlc_renko[ticker]["bar_num"].iloc[i] >= 2 and ohlc_renko[ticker]["obv_slope"].iloc[i] > 30:
                tickers_signal[ticker] = "Buy"
            elif ohlc_renko[ticker]["bar_num"].iloc[i] > -2:
                tickers_signal[ticker] = ""

    ohlc_renko[ticker]["return"] = np.array(tickers_return[ticker])
    

In [41]:
# overall strategys KPIs
strategy_df = pd.DataFrame()

# writing close not return so it works with the functions
for ticker in tickers:
    strategy_df[ticker] = ohlc_renko[ticker]["Close"]

strategy_df["Close"] = strategy_df.mean(axis=1)
print("CAGR: ", cagr(strategy_df))
print("Sharpe: ", sharpe(strategy_df, 0.025))
print("Max DD: ", max_dd(strategy_df))

CAGR:  0.9961313130734342
Sharpe:  2.3303909583970306
Max DD:  1.6564527527404665e-13
